## Section 1 — Chargement et nettoyage (via src/preprocessing.py)

In [1]:
import sys
sys.path.append('..')  # racine du projet, pour importer src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from src.preprocessing import (
    clean_data, get_cat_num_columns, drop_zero_variance,
    group_rare_categories, encode_features
)

# Chargement des données brutes et application de tout le nettoyage déterministe
# (valeurs manquantes, exclusion décès/hospice, regroupement ICD-9, cible binaire)
df = pd.read_csv('../data/raw/diabetic_data.csv')
df = clean_data(df)

print(f"Shape après nettoyage : {df.shape}")

C:\Users\Oumou Salama DAOUDA\AppData\Local\Temp\ipykernel_12256\2655815102.py:17: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/raw/diabetic_data.csv')


Shape après nettoyage : (95673, 47)


In [2]:
import os
print(os.getcwd())

C:\Python_Projects\hospital-readmit-mlops\notebooks


### Nettoyage initial
- Valeurs manquantes ("?") remplacées par NaN
- weight, max_glu_serum, A1Cresult supprimées (>80% de manquants)
- medical_specialty, payer_code → catégorie "Unknown" plutôt que suppression
- race, diag_1/2/3 manquants → lignes supprimées (faible taux, <2%)
- Séjours décès/hospice exclus (discharge_disposition_id 11,13,14,19,20,21) :
  réadmission non pertinente cliniquement pour ces cas
- admission_type_id, discharge_disposition_id, admission_source_id castées en
  catégorielles (codes identifiants, pas des quantités) malgré leur dtype int d'origine

### Regroupement ICD-9
- diag_1/2/3 (700+ modalités chacune) regroupés en 9 catégories cliniques
  (Strack et al., 2014) : Circulatory, Diabetes, Respiratory, Digestive, Injury,
  Musculoskeletal, Genitourinary, Neoplasms, Other
- Codes bruts supprimés après regroupement

## Section 2 — Split train/test par patient_nbr

In [3]:
# Split par patient_nbr (pas par séjour) pour éviter le data leakage
# identifié en EDA (23.5% des patients ont plusieurs séjours)
patients = df['patient_nbr'].unique()
train_patients, test_patients = train_test_split(patients, test_size=0.2, random_state=42)

train_df = df[df['patient_nbr'].isin(train_patients)].copy()
test_df = df[df['patient_nbr'].isin(test_patients)].copy()

overlap = set(train_df['patient_nbr']) & set(test_df['patient_nbr'])
print(f"Train : {train_df.shape[0]} séjours | Test : {test_df.shape[0]} séjours")
print(f"Patients en commun (doit être 0) : {len(overlap)}")
print(f"Taux réadmission train/test : {train_df['readmitted_binary'].mean()*100:.2f}% / {test_df['readmitted_binary'].mean()*100:.2f}%")

X_train = train_df.drop(columns=['readmitted_binary', 'encounter_id', 'patient_nbr'])
y_train = train_df['readmitted_binary']
X_test = test_df.drop(columns=['readmitted_binary', 'encounter_id', 'patient_nbr'])
y_test = test_df['readmitted_binary']

Train : 76689 séjours | Test : 18984 séjours
Patients en commun (doit être 0) : 0
Taux réadmission train/test : 11.50% / 11.61%


### Cible binaire et split
- readmitted (3 modalités) binarisée : <30 = 1 (classe positive), NO/>30 = 0
- Split 80/20 effectué par patient_nbr (pas par séjour) pour éviter le data
  leakage identifié en EDA (23.5% des patients ont plusieurs séjours)
- Vérifié : 0 patient en commun entre train et test
- Déséquilibre préservé dans les deux ensembles (~11.3%)
  
### Séparation features/cible
- encounter_id, patient_nbr retirés des features (identifiants sans valeur
  prédictive, conservés à part pour traçabilité)
- 36 colonnes catégorielles / 8 numériques identifiées séparément

## Section 3 — Préparation de l'encodage (via src/preprocessing.py)


In [4]:
cat_cols, num_cols = get_cat_num_columns(X_train)

# Suppression automatique des colonnes à variance nulle (détectées sur le train)
X_train, X_test, cat_cols, dropped = drop_zero_variance(X_train, X_test, cat_cols)
print(f"Colonnes à variance nulle supprimées : {dropped}")

# Regroupement des catégories rares (<1% du train) — choix assumé ici, pas dans le module
high_card_cols = ['medical_specialty', 'discharge_disposition_id', 'payer_code', 'admission_source_id']
X_train, X_test = group_rare_categories(X_train, X_test, high_card_cols, threshold=0.01)

for col in high_card_cols:
    print(f"{col} : {X_train[col].nunique()} modalités après regroupement")

Colonnes à variance nulle supprimées : ['examide', 'citoglipton', 'metformin-rosiglitazone']
medical_specialty : 11 modalités après regroupement
discharge_disposition_id : 8 modalités après regroupement
payer_code : 11 modalités après regroupement
admission_source_id : 7 modalités après regroupement


### Préparation de l'encodage
- examide, citoglipton, metformin-rosiglitazone supprimées (variance nulle,
  une seule modalité sur tout le train)
- Catégories rares (<1% du train) regroupées en "Other" pour les variables
  à forte cardinalité (medical_specialty 67→11, discharge_disposition_id 21→8,
  payer_code 17→11, admission_source_id 17→7)
- Seuil de regroupement calculé sur le train uniquement, appliqué tel quel
  au test (pas de fuite d'information)

## Section 4 — One-Hot Encoding (via src/preprocessing.py)

In [5]:
import os
os.makedirs('../models', exist_ok=True)

X_train_encoded, X_test_encoded, encoder = encode_features(
    X_train, X_test, cat_cols, num_cols,
    save_path='../models/onehot_encoder.joblib'
)

print(f"Shape X_train_encoded : {X_train_encoded.shape}")
print(f"Shape X_test_encoded  : {X_test_encoded.shape}")

C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [7, 19] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Shape X_train_encoded : (76689, 135)
Shape X_test_encoded  : (18984, 135)


### One-Hot Encoding
- OneHotEncoder (scikit-learn) plutôt que pd.get_dummies : garantit les mêmes
  colonnes en sortie sur train et test (fit sur train, transform sur les deux)
- drop='first' appliqué pour éviter le dummy variable trap (colinéarité
  parfaite par construction si toutes les modalités sont gardées)
- handle_unknown='ignore' : sécurité pour les catégories vues en test mais
  absentes du train (6 et 3 lignes concernées sur medical_specialty/acarbose,
  négligeable)
- Shape final : 168 colonnes (train et test identiques)

## Section 5 — Vérification de la multicolinéarité (VIF)

In [6]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Le VIF (Variance Inflation Factor) mesure à quel point chaque variable
# est expliquée par une combinaison linéaire des autres.
# Règle usuelle : VIF > 10 indique une forte multicolinéarité

vif_data = pd.DataFrame()
vif_data["variable"] = X_train_encoded.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_train_encoded.values, i) 
    for i in range(X_train_encoded.shape[1])
]

print(vif_data.sort_values("VIF", ascending=False).head(20))

                       variable           VIF
72            chlorpropamide_No  37167.867811
103      glyburide-metformin_No  28287.073880
94                  miglitol_No  16717.524859
69               nateglinide_No   7234.619008
66               repaglinide_No   2134.521466
89             rosiglitazone_No   1040.090686
86              pioglitazone_No    718.742081
75               glimepiride_No    495.649287
20                  age_[70-80)    362.388744
19                  age_[60-70)    312.411621
18                  age_[50-60)    244.082869
21                  age_[80-90)    237.549622
104  glyburide-metformin_Steady    191.973367
82                 glyburide_No    171.523872
79                 glipizide_No    160.527910
63                 metformin_No    140.350717
17                  age_[40-50)    136.595324
90         rosiglitazone_Steady     68.556754
87          pioglitazone_Steady     55.402613
16                  age_[30-40)     52.303766


In [7]:
for col in ['chlorpropamide', 'glyburide-metformin', 'miglitol', 'nateglinide']:
    print(f"{col} :")
    print(train_df[col].value_counts())
    print()

chlorpropamide :
chlorpropamide
No        76622
Steady       61
Up            5
Down          1
Name: count, dtype: int64

glyburide-metformin :
glyburide-metformin
No        76167
Steady      514
Up            6
Down          2
Name: count, dtype: int64

miglitol :
miglitol
No        76655
Steady       28
Down          4
Up            2
Name: count, dtype: int64

nateglinide :
nateglinide
No        76129
Steady      532
Up           18
Down         10
Name: count, dtype: int64



### Constat
Deux phénomènes distincts identifiés dans les VIF élevés calculés sur l'ensemble du train (76 689 lignes) :

**1. Dummies issues d'une même variable catégorielle (`age_*`, ex. VIF ~300-500)**
Le VIF classique est structurellement biaisé sur des variables muettes appartenant
à un même groupe catégoriel : connaître toutes les autres dummies d'`age` sauf une
détermine presque entièrement la dernière. Ce n'est pas une vraie redondance avec
d'*autres* variables explicatives, mais une limite connue de l'indicateur lui-même.
L'outil statistiquement correct ici serait le GVIF (Generalized VIF, Fox & Monette 1992),
non disponible nativement dans statsmodels pour du one-hot.

**2. Médicaments très rarement prescrits (`chlorpropamide_No`, VIF > 30 000)**
La modalité de référence retirée par `drop='first'` (`Down`) ne représente que
1 à 10 occurrences sur 76 689 séjours selon le médicament. Ce déséquilibre extrême
rend la modalité restante quasi parfaitement déterminée par l'intercept et les
autres dummies — un artefact d'encodage sur des catégories rares, pas une
redondance clinique réelle.

### Décision retenue
- Aucune variable supprimée : la régularisation L2 par défaut de scikit-learn
  stabilise numériquement l'estimation malgré cette colinéarité apparente
- Les coefficients individuels des médicaments très rares ne seront pas
  surinterprétés dans l'analyse finale (J9, SHAP) — on privilégiera leur
  contribution agrégée si besoin
- Point à mentionner explicitement dans le README / en entretien comme
  limite méthodologique assumée, plutôt que dissimulée

## Section 6 — Baseline Logistic Regression (sans rééquilibrage) + tracking MLflow

In [8]:
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, 
    classification_report, confusion_matrix
)

# Création/sélection de l'expérience MLflow dans laquelle ce run sera tracké
mlflow.set_experiment("hospital-readmit-baseline")

# Chaque "run" MLflow regroupe les paramètres, métriques et le modèle d'un seul entraînement
with mlflow.start_run(run_name="logreg_no_rebalancing"):

    # Baseline : Logistic Regression sans aucune gestion du déséquilibre
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_train_encoded, y_train)

    # Prédictions sur le test : classes (0/1) et probabilités (utiles pour AUC)
    y_pred = model.predict(X_test_encoded)
    y_pred_proba = model.predict_proba(X_test_encoded)[:, 1]

    # AUC : capacité globale à distinguer les deux classes
    # PR-AUC : plus informatif que l'AUC sur un dataset déséquilibré (se concentre sur la classe positive)
    auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)

    # Enregistrement des paramètres et métriques dans MLflow, pour pouvoir comparer ce run aux suivants
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("rebalancing", "none")
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)

    # Sauvegarde du modèle entraîné lui-même (rechargeable plus tard depuis MLflow)
    mlflow.sklearn.log_model(model, "model")

    print(f"AUC      : {auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    # classification_report : precision/recall/f1 par classe — essentiel ici pour voir
    # si le modèle détecte vraiment la classe minoritaire (réadmissions), pas juste l'accuracy globale
    print(f"\n{classification_report(y_test, y_pred)}")
    # Matrice de confusion : compte exact des vrais/faux positifs et négatifs
    print(f"\nMatrice de confusion :\n{confusion_matrix(y_test, y_pred)}")

C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/06/22 14:14:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


AUC      : 0.6576
PR-AUC   : 0.2151

              precision    recall  f1-score   support

           0       0.89      1.00      0.94     16780
           1       0.57      0.01      0.02      2204

    accuracy                           0.88     18984
   macro avg       0.73      0.51      0.48     18984
weighted avg       0.85      0.88      0.83     18984


Matrice de confusion :
[[16759    21]
 [ 2176    28]]


## Section 6bis — Comparaison L2 vs L1 (Lasso)

In [9]:
from sklearn.linear_model import LogisticRegression

# Deux configurations à comparer : L2 (par défaut, ne pousse jamais les coefficients
# à zéro) vs L1/Lasso (pousse certains coefficients exactement à zéro, sélection de variables)
# liblinear est nécessaire pour L1 : le solver par défaut (lbfgs) ne le supporte pas
configs = [
    {"name": "logreg_l2", "penalty": "l2", "solver": "lbfgs"},
    {"name": "logreg_l1", "penalty": "l1", "solver": "liblinear"},
]

results = {}

for cfg in configs:
    # Un run MLflow distinct par configuration, pour pouvoir les comparer dans l'UI MLflow
    with mlflow.start_run(run_name=cfg["name"]):

        model = LogisticRegression(
            penalty=cfg["penalty"], 
            solver=cfg["solver"], 
            max_iter=1000, 
            random_state=42
        )
        model.fit(X_train_encoded, y_train)

        y_pred_proba = model.predict_proba(X_test_encoded)[:, 1]
        auc = roc_auc_score(y_test, y_pred_proba)
        pr_auc = average_precision_score(y_test, y_pred_proba)

        # Nombre de coefficients réellement à zéro (pertinent surtout pour L1 :
        # L2 ne produit quasiment jamais de zéro exact)
        n_zero_coefs = (model.coef_[0] == 0).sum()

        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("penalty", cfg["penalty"])
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("pr_auc", pr_auc)
        mlflow.log_metric("n_zero_coefs", n_zero_coefs)
        mlflow.sklearn.log_model(model, "model")

        # Résultats gardés en mémoire dans le notebook, pour comparaison rapide sans
        # avoir à rouvrir l'UI MLflow
        results[cfg["name"]] = {"auc": auc, "pr_auc": pr_auc, "n_zero_coefs": n_zero_coefs}

        print(f"{cfg['name']:12s} | AUC : {auc:.4f} | PR-AUC : {pr_auc:.4f} | Coefs à 0 : {n_zero_coefs}/{len(model.coef_[0])}")

C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://s

logreg_l2    | AUC : 0.6576 | PR-AUC : 0.2151 | Coefs à 0 : 0/135


C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
2026/06/22 14:16:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


logreg_l1    | AUC : 0.6578 | PR-AUC : 0.2154 | Coefs à 0 : 21/135


### Comparaison L2 vs L1 (Lasso)

**Principe**
La régularisation pénalise la magnitude des coefficients pour éviter le
surajustement. L2 (Ridge, par défaut) pénalise la somme des carrés des
coefficients — elle les rétrécit tous vers zéro sans jamais les annuler
complètement. L1 (Lasso) pénalise la somme des valeurs absolues — propriété
mathématique qui pousse certains coefficients exactement à zéro, réalisant
ainsi une vraie sélection de variables intégrée à l'entraînement.

**Pourquoi L1 est préférable au stepwise AIC pour cette sélection**
Le stepwise sélectionne les variables via des tests successifs sur les mêmes
données, ce qui biaise les coefficients et les intervalles de confiance du
modèle final (la sélection n'est pas prise en compte dans l'inférence
statistique qui suit). L1 réalise cette sélection de façon continue et
pénalisée en une seule estimation, sans ce biais post-sélection, et reste
peu coûteux même avec un grand nombre de variables (135 ici).

**Résultat obtenu**
| | AUC | PR-AUC | Coefs à 0 |
|---|---|---|---|
| L2 | 0.6576 | 0.2151 | 0/135 |
| L1 | 0.6578 | 0.2154 | 21/135 |

Performance quasi identique, mais L1 élimine ~15% des variables (21/135)
sans aucune perte prédictive → modèle plus parcimonieux et plus interprétable,
retenu comme référence pour la suite du projet.

## Section 6ter — Variables éliminées par la régularisation L1

In [10]:
# Récupération du modèle L1 (à relancer si non conservé en mémoire depuis la boucle)
model_l1 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42)
model_l1.fit(X_train_encoded, y_train)

# Association de chaque coefficient à son nom de variable, pour pouvoir les identifier
coefs = pd.Series(model_l1.coef_[0], index=X_train_encoded.columns)

# Variables que L1 a jugées non-informatives une fois les autres dans le modèle
zero_coefs = coefs[coefs == 0]
print(f"Variables à coefficient nul ({len(zero_coefs)}) :")
print(zero_coefs.index.tolist())

# Valeur absolue : on s'intéresse à l'ampleur de l'effet, peu importe le sens (+/-)
# Attention : un coefficient élevé sur une variable rare peut être instable plutôt
# que réellement informatif (cf. Section 7bis, VIF) — à ne pas surinterpréter ici
print(f"\n--- Pour comparaison, les 10 coefficients les plus forts (en valeur absolue) ---")
print(coefs.abs().sort_values(ascending=False).head(10))

C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Variables à coefficient nul (21) :
['race_Asian', 'gender_Unknown/Invalid', 'age_[20-30)', 'admission_type_id_4', 'admission_type_id_8', 'repaglinide_Up', 'chlorpropamide_Up', 'glimepiride_Up', 'acetohexamide_Steady', 'rosiglitazone_Up', 'acarbose_Up', 'miglitol_Up', 'troglitazone_Steady', 'tolazamide_Steady', 'tolazamide_Up', 'glyburide-metformin_Up', 'glipizide-metformin_Steady', 'glimepiride-pioglitazone_Steady', 'metformin-pioglitazone_Steady', 'diag_2_cat_Other', 'diag_3_cat_Musculoskeletal']

--- Pour comparaison, les 10 coefficients les plus forts (en valeur absolue) ---
discharge_disposition_id_22    1.329021
discharge_disposition_id_5     0.960659
chlorpropamide_Steady          0.881960
miglitol_Steady                0.820466
discharge_disposition_id_2     0.587439
metformin_Up                   0.496275
age_[10-20)                    0.458390
diag_2_cat_Neoplasms           0.437258
nateglinide_Up                 0.435556
glipizide_No                   0.413808
dtype: float64


### Variables éliminées et coefficients dominants (L1)

21 variables à coefficient nul, très majoritairement des catégories rares
(médicaments peu prescrits, modalités démographiques à faible effectif) —
cohérent avec les VIF élevés identifiés en Section 7bis.

Attention : les coefficients les plus élevés en valeur absolue incluent
plusieurs de ces mêmes catégories rares (ex. chlorpropamide_Steady,
miglitol_Steady) — magnitude élevée mais peu fiable statistiquement
(peu d'observations). Ne pas surinterpréter sans intervalle de confiance
(cf. J8, bootstrap). Les coefficients sur discharge_disposition_id
(transferts vers soins de suite) reposent sur des effectifs plus solides
et sont cliniquement plausibles.

## Section 7 — Comparaison des stratégies de rééquilibrage

### Stratégie 1 : class weights
Pondère la fonction de perte pour compenser le déséquilibre, sans toucher
aux données. `class_weight='balanced'` ajuste automatiquement le poids de
chaque classe inversement proportionnel à sa fréquence.

In [11]:
# On garde L1 comme référence (retenue en Section 8bis), et on ajoute class_weight='balanced' :
# pondère la fonction de perte pour pénaliser davantage les erreurs sur la classe minoritaire,
# sans modifier les données elles-mêmes (contrairement à SMOTE, comparé juste après)
model_cw = LogisticRegression(
    penalty='l1', solver='liblinear', class_weight='balanced',
    max_iter=1000, random_state=42
)

with mlflow.start_run(run_name="logreg_l1_class_weight"):
    model_cw.fit(X_train_encoded, y_train)

    y_pred = model_cw.predict(X_test_encoded)
    y_pred_proba = model_cw.predict_proba(X_test_encoded)[:, 1]

    auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("penalty", "l1")
    mlflow.log_param("rebalancing", "class_weight")
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.sklearn.log_model(model_cw, "model")

    print(f"AUC      : {auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    # À comparer au rappel quasi nul (0.01) obtenu sans rééquilibrage (Section 8) :
    # on attend une nette hausse du rappel sur la classe positive, au prix probable
    # d'une baisse de la precision (compromis attendu, pas un problème en soi)
    print(f"\n{classification_report(y_test, y_pred)}")

C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
2026/06/22 14:22:11 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


AUC      : 0.6591
PR-AUC   : 0.2156

              precision    recall  f1-score   support

           0       0.92      0.69      0.79     16780
           1       0.18      0.53      0.27      2204

    accuracy                           0.67     18984
   macro avg       0.55      0.61      0.53     18984
weighted avg       0.83      0.67      0.73     18984



### Class weights

**Principe**
class_weight='balanced' pondère la fonction de perte de la Logistic
Regression : les erreurs sur la classe minoritaire (réadmissions) sont
pénalisées proportionnellement plus que celles sur la classe majoritaire.
Aucune modification des données, juste un changement de l'objectif optimisé.

**Résultat obtenu (comparé à la baseline L1 sans rééquilibrage)**
| | Sans rééquilibrage | Class weights |
|---|---|---|
| Rappel classe 1 | 0.01 | 0.53 |
| Precision classe 1 | 0.57 | 0.18 |
| AUC | 0.6576 | 0.6591 |
| PR-AUC | 0.2151 | 0.2156 |
| Accuracy | 0.88 | 0.67 |

Le rappel passe de quasi nul à 53% : le modèle devient réellement utile pour
détecter les réadmissions, au prix d'une chute de precision (beaucoup de
faux positifs) et d'accuracy globale — attendu et non problématique, puisque
l'accuracy n'est pas une métrique pertinente sur un

### Stratégie 2 : SMOTE
Plutôt que de pondérer la perte, SMOTE génère des observations synthétiques
de la classe minoritaire par interpolation entre voisins proches (k-NN dans
l'espace des features), pour rééquilibrer artificiellement le train.

Point de vigilance statistique : SMOTE a été conçu à l'origine pour des
variables continues. Appliqué ici sur une matrice entièrement one-hot
encodée, l'interpolation peut générer des valeurs fractionnaires (ex. 0.37)
sur des colonnes binaires qui ne devraient valoir que 0 ou 1 — une limite
connue, qu'on documente plutôt que d'ignorer. L'alternative plus rigoureuse
serait SMOTENC (gère nativement un mélange numérique/catégoriel), à explorer
si SMOTE classique pose problème.

Règle absolue : SMOTE s'applique uniquement sur le train, jamais sur le test
(les observations synthétiques ne doivent jamais contaminer l'évaluation).

In [12]:
from imblearn.over_sampling import SMOTE

# random_state pour la reproductibilité de la génération synthétique
smote = SMOTE(random_state=42)

# fit_resample uniquement sur le train : génère des observations synthétiques
# de la classe minoritaire pour équilibrer les deux classes à 50/50
X_train_smote, y_train_smote = smote.fit_resample(X_train_encoded, y_train)

print(f"Avant SMOTE : {y_train.value_counts().to_dict()}")
print(f"Après SMOTE : {y_train_smote.value_counts().to_dict()}")

model_smote = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000, random_state=42)

with mlflow.start_run(run_name="logreg_l1_smote"):
    model_smote.fit(X_train_smote, y_train_smote)

    # Évaluation sur le test ORIGINAL (jamais transformé par SMOTE)
    y_pred = model_smote.predict(X_test_encoded)
    y_pred_proba = model_smote.predict_proba(X_test_encoded)[:, 1]

    auc = roc_auc_score(y_test, y_pred_proba)
    pr_auc = average_precision_score(y_test, y_pred_proba)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("penalty", "l1")
    mlflow.log_param("rebalancing", "smote")
    mlflow.log_metric("auc", auc)
    mlflow.log_metric("pr_auc", pr_auc)
    mlflow.sklearn.log_model(model_smote, "model")

    print(f"AUC      : {auc:.4f}")
    print(f"PR-AUC   : {pr_auc:.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")
    print(f"\nMatrice de confusion :\n{confusion_matrix(y_test, y_pred)}")

Avant SMOTE : {0: 67870, 1: 8819}
Après SMOTE : {0: 67870, 1: 67870}


C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Python_Projects\hospital-readmit-mlops\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
2026/06/22 14:31:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


AUC      : 0.6213
PR-AUC   : 0.1881

              precision    recall  f1-score   support

           0       0.91      0.63      0.74     16780
           1       0.16      0.55      0.25      2204

    accuracy                           0.62     18984
   macro avg       0.54      0.59      0.50     18984
weighted avg       0.83      0.62      0.69     18984


Matrice de confusion :
[[10534  6246]
 [  993  1211]]


### SMOTE et décision finale de rééquilibrage

**Résultat (comparé aux deux approches précédentes)**
| | Sans rééquilibrage | Class weights | SMOTE |
|---|---|---|---|
| Rappel classe 1 | 0.01 | 0.53 | 0.55 |
| Precision classe 1 | 0.57 | 0.18 | 0.16 |
| AUC | 0.6576 | 0.6591 | 0.6213 |
| PR-AUC | 0.2151 | 0.2156 | 0.1881 |

SMOTE n'apporte pas de gain de rappel significatif par rapport à class
weights, et dégrade l'AUC/PR-AUC — probablement lié à l'interpolation
SMOTE classique sur une matrice presque entièrement one-hot (génère des
valeurs fractionnaires sur des colonnes binaires, peu de sens géométrique
dans cet espace). SMOTENC n'a pas été testé ici par souci de temps, mais
serait l'alternative à explorer si on revisitait cette comparaison.

**Décision retenue pour la suite du projet**
Logistic Regression, pénalité L1, class_weight='balanced' — modèle de
référence pour la comparaison aux modèles avancés (J4) et pour les analyses
de métriques approfondies (J5-J9).

### Rappel — Interprétation de la precision et du rappel

**Precision (classe 1)** : parmi les patients prédits "à risque", quelle
proportion l'est réellement.
Precision = VP / (VP + FP)
→ "Quand le modèle sonne l'alarme, a-t-il raison ?"

**Rappel / Recall (classe 1)** : parmi les patients réellement réadmis,
quelle proportion le modèle a détectée.
Rappel = VP / (VP + FN)
→ "Sur tous les vrais cas à risque, combien le modèle en a-t-il trouvé ?"

**Application sur le modèle class weights**
Matrice de confusion :
              Prédit 0    Prédit 1
Réel 0        11554       5226
Réel 1        1037        1167

- Rappel = 1167 / (1167 + 1037) = 0.53 → 53% des réadmissions réelles détectées,
  1037 manquées (faux négatifs)
- Precision = 1167 / (1167 + 5226) = 0.18 → 18% des alertes sont de vraies
  réadmissions, 5226 fausses alertes (faux positifs)

**Compromis precision/rappel**
Les deux évoluent en sens inverse : un modèle plus "prudent" (alerte sur plus
de patients) détecte plus de vrais cas (rappel ↑) mais multiplie les fausses
alertes (precision ↓). Confirmé empiriquement : sans rééquilibrage
(rappel 0.01, precision 0.57) → class weights (rappel 0.53, precision 0.18).

**Pourquoi privilégier le rappel ici**
Un faux négatif = réadmission évitable manquée (coût humain et financier
direct). Un faux positif = vérification supplémentaire (coût plus faible).
Justifie de tolérer une precision modeste en échange d'un rappel élevé —
le compromis exact sera formalisé au J6 (Decision Curve Analysis) plutôt
que laissé implicite au seuil par défaut de 0.5.